# Block Drop — Hive Server Analytics

**Block Drop** is a minigame on [The Hive](https://playhive.com/), one of the most popular Minecraft Bedrock servers.  
In Block Drop, players stand on a platform of blocks that gradually disappears beneath them — the last player standing wins.  
Strategy involves collecting powerups, using vaults to reposition, and destroying blocks to eliminate opponents.

---

## Project Overview

This notebook analyzes leaderboard data scraped from the top 1000 ranked Block Drop players on the Hive.  
The goal is to understand **what drives win rate** using regression and ensemble models.

### Features
| Feature | Description |
|---|---|
| `BlocksPerGame` | Average blocks destroyed per game — proxy for aggression / map control |
| `PowerupsPerGame` | Average powerups collected per game — powerups provide in-game advantages |
| `VaultsPerGame` | Average vaults used per game — vaults allow players to double jump to escape death and go to higher levels |

### Target
| Variable | Description |
|---|---|
| `WinPercentage` | Proportion of games won (0–1) |

### Models Used
1. **Linear Regression** — baseline interpretable model
2. **Ridge Regression** — linear model with L2 regularization to reduce overfitting
3. **Cross-Validated Feature Analysis** — isolates each feature's individual predictive power
4. **Random Forest** — ensemble of decision trees, captures non-linear relationships
5. **AdaBoost** — boosting ensemble that iteratively corrects residual errors

---
## 1. Setup & Data Loading

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

# ── Load Data ──────────────────────────────────────────────────────────────────
# Dataset: top 999 Block Drop players scraped from the Hive leaderboard.
# Per-game rate features (BlocksPerGame, PowerupsPerGame, VaultsPerGame) are
# pre-computed as raw_stat / games_played to normalize for players with
# vastly different game counts.
df = pd.read_csv("blockdrop.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nBasic stats:")
df[['WinPercentage', 'BlocksPerGame', 'PowerupsPerGame', 'VaultsPerGame']].describe().round(3)

Dataset shape: (1000, 12)

Column names: ['Username', 'Rank', 'Played', 'Victories', 'WinPercentage', 'Deaths', 'Blocks Destroyed', 'Powerups Collected', 'Vaults Used', 'BlocksPerGame', 'PowerupsPerGame', 'VaultsPerGame']

Basic stats:


,WinPercentage,BlocksPerGame,PowerupsPerGame,VaultsPerGame
count,1000.000,1000.000,1000.000,1000.000
mean,0.557,792.543,3.245,1.381
std,0.208,144.597,1.141,0.691
min,0.060,51.939,0.118,0.003
25%,0.400,703.496,2.479,0.881
50%,0.540,786.338,3.141,1.354
75%,0.710,886.336,4.000,1.792
max,1.000,1354.848,6.948,3.527


---
## 2. Feature / Target Split & Train-Test Split

In [2]:
# ── Features & Target ─────────────────────────────────────────────────────────
# We use three per-game rate statistics as predictors of WinPercentage.
# Raw cumulative stats (total blocks, total powerups) are NOT used directly
# because they correlate with games played, not skill.
FEATURES = ['BlocksPerGame', 'PowerupsPerGame', 'VaultsPerGame']
TARGET   = 'WinPercentage'

X = df[FEATURES]
y = df[TARGET]

# ── Train / Test Split ────────────────────────────────────────────────────────
# 80/20 split with random_state=42 for reproducibility.
# Stratification is not applied — WinPercentage is continuous.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

Training samples : 800
Test samples     : 200


---
## 3. Linear Regression & Ridge Regression

We start with linear models as an interpretable baseline.  
**Ridge** adds L2 regularization (penalizes large coefficients), which helps when features are mildly correlated — it shouldn't change results much here but is a good sanity check.

In [3]:
# ── Linear Regression ─────────────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("═" * 40)
print("LINEAR REGRESSION")
print("═" * 40)
print(f"R²   : {r2_score(y_test, lr_preds):.4f}")
print(f"RMSE : {root_mean_squared_error(y_test, lr_preds):.4f}")
print()
print("Coefficients (effect on WinPercentage per 1-unit increase):")
for feat, coef in zip(FEATURES, lr.coef_):
    direction = "↑" if coef > 0 else "↓"
    print(f"  {feat:<20} {coef:+.10f}  {direction}")
print(f"  Intercept            {lr.intercept_:.4f}")

# ── Ridge Regression ──────────────────────────────────────────────────────────
# alpha=1.0 is the default regularization strength.
# If Ridge and OLS produce nearly identical coefficients (as here),
# it confirms the features are not badly multicollinear.
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_preds = ridge.predict(X_test)

print()
print("═" * 40)
print("RIDGE REGRESSION  (alpha=1.0)")
print("═" * 40)
print(f"R²   : {r2_score(y_test, ridge_preds):.4f}")
print(f"RMSE : {root_mean_squared_error(y_test, ridge_preds):.4f}")
print()
print("Coefficients:")
for feat, coef in zip(FEATURES, ridge.coef_):
    direction = "↑" if coef > 0 else "↓"
    print(f"  {feat:<20} {coef:+.10f}  {direction}")

print()
print("Note: Ridge and OLS coefficients are nearly identical → low multicollinearity.")

════════════════════════════════════════
LINEAR REGRESSION
════════════════════════════════════════
R²   : 0.5550
RMSE : 0.1311

Coefficients (effect on WinPercentage per 1-unit increase):
  BlocksPerGame        +0.0000270210  ↑
  PowerupsPerGame      +0.1338650165  ↑
  VaultsPerGame        -0.0908058407  ↓
  Intercept            0.2270

════════════════════════════════════════
RIDGE REGRESSION  (alpha=1.0)
════════════════════════════════════════
R²   : 0.5553
RMSE : 0.1311

Coefficients:
  BlocksPerGame        +0.0000279162  ↑
  PowerupsPerGame      +0.1336847175  ↑
  VaultsPerGame        -0.0905616230  ↓

Note: Ridge and OLS coefficients are nearly identical → low multicollinearity.


---
## 4. Cross-Validated Feature Analysis

To understand how much each feature contributes **on its own**, we fit a simple linear model for each feature individually with 5-fold CV.  
This isolates each predictor's standalone signal and is more reliable than a single train/test split.

In [4]:
# ── Individual Feature CV Analysis ────────────────────────────────────────────
# For each feature, we fit a univariate linear regression using 5-fold CV
# and report mean R² across folds. Higher = stronger standalone predictor.
print("═" * 40)
print("UNIVARIATE CV ANALYSIS  (5-fold)")
print("═" * 40)

results = {}
for feature in FEATURES:
    X_single = df[[feature]]
    scores = cross_val_score(
        LinearRegression(),
        X_single,
        y,
        cv=5,
        scoring="r2"
    )
    results[feature] = scores.mean()
    print(f"\n{feature}")
    print(f"  Mean CV R² : {scores.mean():.4f}")
    print(f"  Fold scores: {[round(s, 4) for s in scores]}")

print()
best = max(results, key=results.get)
print(f"→ Strongest individual predictor: {best} (R² = {results[best]:.4f})")
print("→ PowerupsPerGame alone explains ~55% of variance in WinPercentage.")
print("→ BlocksPerGame and VaultsPerGame add modest incremental value.")

════════════════════════════════════════
UNIVARIATE CV ANALYSIS  (5-fold)
════════════════════════════════════════

BlocksPerGame
  Mean CV R² : 0.1836
  Fold scores: [np.float64(0.2098), np.float64(0.2416), np.float64(0.0077), np.float64(0.215), np.float64(0.2437)]

PowerupsPerGame
  Mean CV R² : 0.5460
  Fold scores: [np.float64(0.4585), np.float64(0.6117), np.float64(0.4811), np.float64(0.5596), np.float64(0.6189)]

VaultsPerGame
  Mean CV R² : 0.0975
  Fold scores: [np.float64(0.1574), np.float64(0.1019), np.float64(0.1119), np.float64(0.1081), np.float64(0.0082)]

→ Strongest individual predictor: PowerupsPerGame (R² = 0.5460)
→ PowerupsPerGame alone explains ~55% of variance in WinPercentage.
→ BlocksPerGame and VaultsPerGame add modest incremental value.


---
## 5. Ensemble Models — Random Forest & AdaBoost

Linear models assume a straight-line relationship between features and WinPercentage.  
Ensemble tree models can capture **non-linear patterns** (e.g., powerup collection only helps up to a point, or interacts with vaults).  

- **Random Forest**: builds many deep trees independently, each on a random subset of data + features, then averages predictions. Reduces variance.
- **AdaBoost**: builds shallow trees sequentially, each one focusing more on the mistakes of the previous. Reduces bias.

In [5]:
# ── Random Forest ─────────────────────────────────────────────────────────────
# n_estimators=100 : 100 trees in the forest
# max_features=2   : each tree considers 2 of the 3 features at each split
#                    (introduces diversity among trees, reduces correlation)
# random_state=42  : reproducibility
rf = RandomForestRegressor(
    n_estimators=100,
    max_features=2,
    random_state=42
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Feature importance: how often each feature is used in splits, weighted
# by how much it reduces impurity (MSE for regressors)
rf_importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("═" * 40)
print("RANDOM FOREST")
print("═" * 40)
print(f"R²   : {r2_score(y_test, rf_preds):.6f}")
print(f"RMSE : {root_mean_squared_error(y_test, rf_preds):.6f}")
print(f"MAE  : {mean_absolute_error(y_test, rf_preds):.6f}")
print()
print("Feature Importances (Gini-based):")
print(rf_importance.to_string(index=False))

# ── AdaBoost ──────────────────────────────────────────────────────────────────
# estimator      : shallow decision tree (max_depth=3) as the base learner
# n_estimators   : 1000 boosting rounds — more rounds with a low learning rate
#                  is the standard AdaBoost tuning pattern
# learning_rate  : 0.05 — shrinks each tree's contribution; prevents overfitting
#                  when paired with many estimators
ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42
)
ada.fit(X_train, y_train)
ada_preds = ada.predict(X_test)

ada_importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': ada.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print()
print("═" * 40)
print("ADABOOST  (1000 rounds, lr=0.05)")
print("═" * 40)
print(f"R²   : {r2_score(y_test, ada_preds):.6f}")
print(f"RMSE : {root_mean_squared_error(y_test, ada_preds):.6f}")
print(f"MAE  : {mean_absolute_error(y_test, ada_preds):.6f}")
print()
print("Feature Importances:")
print(ada_importance.to_string(index=False))

════════════════════════════════════════
RANDOM FOREST
════════════════════════════════════════
R²   : 0.667742
RMSE : 0.113328
MAE  : 0.089752

Feature Importances (Gini-based):
        Feature  Importance
PowerupsPerGame    0.601080
  VaultsPerGame    0.208983
  BlocksPerGame    0.189937

════════════════════════════════════════
ADABOOST  (1000 rounds, lr=0.05)
════════════════════════════════════════
R²   : 0.669334
RMSE : 0.113057
MAE  : 0.089337

Feature Importances:
        Feature  Importance
PowerupsPerGame    0.598366
  VaultsPerGame    0.234705
  BlocksPerGame    0.166929



════════════════════════════════════════
ADABOOST  (1000 rounds, lr=0.05)
════════════════════════════════════════
R²   : 0.669334
RMSE : 0.113057
MAE  : 0.089337

Feature Importances:
        Feature  Importance
PowerupsPerGame    0.598366
  VaultsPerGame    0.234705
  BlocksPerGame    0.166929


---
## 6. Model Comparison & Key Takeaways

In [6]:
# ── Summary Table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Random Forest', 'AdaBoost'],
    'R²':    [
        round(r2_score(y_test, lr_preds), 4),
        round(r2_score(y_test, ridge_preds), 4),
        round(r2_score(y_test, rf_preds), 4),
        round(r2_score(y_test, ada_preds), 4),
    ],
    'RMSE':  [
        round(root_mean_squared_error(y_test, lr_preds), 4),
        round(root_mean_squared_error(y_test, ridge_preds), 4),
        round(root_mean_squared_error(y_test, rf_preds), 4),
        round(root_mean_squared_error(y_test, ada_preds), 4),
    ]
})

print("MODEL COMPARISON")
print(summary.to_string(index=False))

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY TAKEAWAYS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. PowerupsPerGame is by far the strongest predictor of win rate.
   It accounts for ~55% of variance on its own (univariate CV R² = 0.546)
   and ~60% of feature importance in both ensemble models.
   → High-win-rate players are consistently collecting more powerups per game.

2. VaultsPerGame has a NEGATIVE coefficient in the linear model.
   This seems counterintuitive but may reflect that lower-ranked players
   (who have lower win rates) use vaults more defensively / reactively,
   while elite players win before needing them.

3. BlocksPerGame has a positive but very small linear coefficient,
   yet non-trivial importance (~17-19%) in the ensemble models.
   This suggests a non-linear relationship: block destruction matters,
   but only up to a threshold.

4. Ensemble models (RF, AdaBoost) outperform linear models by ~11% R²,
   confirming that the feature-win relationships are not purely linear.
   Both ensembles agree closely on feature importance rankings.
""")

MODEL COMPARISON
            Model     R²   RMSE
Linear Regression 0.5550 0.1311
 Ridge Regression 0.5553 0.1311
    Random Forest 0.6677 0.1133
         AdaBoost 0.6693 0.1131

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY TAKEAWAYS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. PowerupsPerGame is by far the strongest predictor of win rate.
   It accounts for ~55% of variance on its own (univariate CV R² = 0.546)
   and ~60% of feature importance in both ensemble models.
   → High-win-rate players are consistently collecting more powerups per game.

2. VaultsPerGame has a NEGATIVE coefficient in the linear model.
   This seems counterintuitive but may reflect that lower-ranked players
   (who have lower win rates) use vaults more defensively / reactively,
   while elite players win before needing them.

3. BlocksPerGame has a positive but very small linear coefficient,
   yet non-trivial importance (~17-19%) in the ensemble models.
   This suggests a non-linear relationship: block destru